# ΔRH6Ng1 — деплой сайта в Jupyter

Этот ноутбук собирает сайт стратегии **ΔRH6Ng1** и поднимает его прямо внутри Jupyter: последняя ячейка показывает работающий сайт во фрейме.

Работает в **Google Colab** и в обычном **Jupyter Lab / Notebook**.

Порядок: выполняйте ячейки сверху вниз (`Shift+Enter`).

Что происходит:
1. находится каталог с сайтом (или репозиторий клонируется с GitHub);
2. проверяется Node.js ≥ 20 (в Colab ставится автоматически);
3. `npm install` + `npm run build` → статический сайт в `out/`;
4. запускается локальный HTTP-сервер;
5. сайт отображается в ячейке ноутбука.

In [ ]:
# 1. Каталог с сайтом: текущий, если ноутбук лежит в репозитории, иначе — клонируем
import pathlib
import subprocess

REPO_URL = "https://github.com/BazanovGo/Site.git"  # для приватного репо: https://<TOKEN>@github.com/BazanovGo/Site.git
BRANCH = "claude/drh6ng1-strategy-site-aqfd8a"
PORT = 8020

if pathlib.Path("package.json").exists():
    SITE_DIR = pathlib.Path.cwd()
else:
    SITE_DIR = pathlib.Path.cwd() / "Site"
    if not SITE_DIR.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(SITE_DIR)],
            check=True,
        )

print("Каталог сайта:", SITE_DIR)

In [ ]:
# 2. Node.js >= 20 (в Colab при необходимости ставится сам)
import shutil

def node_major() -> int:
    if not shutil.which("node"):
        return 0
    version = subprocess.check_output(["node", "-v"], text=True).strip()
    return int(version.lstrip("v").split(".")[0])

if node_major() < 20:
    print("Устанавливаю Node.js 22 …")
    subprocess.run(
        "curl -fsSL https://deb.nodesource.com/setup_22.x | bash - "
        "&& apt-get install -y nodejs",
        shell=True,
        check=True,
    )

print("node", subprocess.check_output(["node", "-v"], text=True).strip())
print("npm ", subprocess.check_output(["npm", "-v"], text=True).strip())

In [ ]:
# 3. Сборка: зависимости + статический экспорт Next.js в out/
subprocess.run(["npm", "install", "--no-audit", "--no-fund"], cwd=SITE_DIR, check=True)
subprocess.run(["npm", "run", "build"], cwd=SITE_DIR, check=True)

OUT_DIR = SITE_DIR / "out"
assert (OUT_DIR / "index.html").exists(), "Сборка не создала out/index.html"
print("Готово:", OUT_DIR)

In [ ]:
# 4. Локальный HTTP-сервер со статикой (фоновый поток)
import functools
import http.server
import threading

handler = functools.partial(
    http.server.SimpleHTTPRequestHandler, directory=str(OUT_DIR)
)

try:
    http.server.ThreadingHTTPServer.allow_reuse_address = True
    httpd = http.server.ThreadingHTTPServer(("127.0.0.1", PORT), handler)
    threading.Thread(target=httpd.serve_forever, daemon=True).start()
    print(f"Сервер запущен: http://127.0.0.1:{PORT}")
except OSError:
    print(f"Порт {PORT} уже занят — вероятно, сервер запущен ранее. Продолжаем.")

In [ ]:
# 5. Показать сайт прямо в ноутбуке
from IPython.display import IFrame, display

try:
    from google.colab.output import eval_js  # noqa: в Colab фрейм идёт через прокси
    url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
except ImportError:
    url = f"http://127.0.0.1:{PORT}/"

print("Сайт доступен по адресу:", url)
display(IFrame(url, width="100%", height=820))

---
Сервер живёт, пока работает ядро ноутбука. Чтобы пересобрать сайт после изменений — выполните ячейки 3–5 заново.

*Информация на сайте не является индивидуальной инвестиционной рекомендацией. Торговля фьючерсами связана с высоким риском потери капитала.*